# 02 Preprocessing


## CRISP-DM Stage
Data Preparation.

## Research Context
Phase 7A prepares the completed Google Play raw review dataset for downstream IndoBERT sentiment modeling and SVM aspect classification. This phase does not train models, fit TF-IDF, download model artifacts, or split train/validation/test data.

## Preprocessing Objective
The objective is to preserve the original `initial_sentiment`, create a conservative corrected `final_sentiment`, and produce two text representations: `text_indobert` for IndoBERT and `text_svm` for later SVM feature extraction.

## Input Dataset
Primary input:
- `datasets/raw/reviews_raw_labeled.csv`

Expected columns include `source`, `app_id`, `external_id`, `author_name`, `rating`, `content`, `likes`, `app_version`, `reviewed_at`, `scraped_at`, `sort_method`, and `initial_sentiment`. Optional columns are handled safely by the scripts.

## Initial Sentiment Label Review
`initial_sentiment` is derived from star rating in the acquisition phase and must remain unchanged. It is retained as an audit column for comparing rating-based labels against the keyword-corrected `final_sentiment`.

## Keyword-based Relabeling Strategy
Relabeling is intentionally conservative. Neutral rows may change to `Negative` or `Positive` only when a one-direction strong keyword signal appears. Mixed signals are not relabeled and are marked as audit candidates through `relabel_reason`. Rating 1, 2, 4, and 5 rows are not aggressively relabeled; strong contradictions are documented for audit only.

## IndoBERT Preprocessing Strategy
`text_indobert` uses conservative text cleaning: Unicode normalization, URL removal, HTML tag/entity cleanup, zero-width space cleanup, and whitespace normalization. It preserves sentence structure, negation words, punctuation where possible, and emoji where possible. It does not apply stemming or stopword removal.

## SVM Preprocessing Strategy
`text_svm` uses stronger normalization suitable for later feature extraction: lowercasing, URL removal, HTML cleanup, conservative repeated-character normalization, punctuation cleanup, and simple emoji token handling. It does not fit TF-IDF, create vectorizer artifacts, stem text, or split the dataset.

## Negation Preservation Note
Negation words such as `tidak`, `bukan`, `belum`, `jangan`, `ga`, `gak`, and `nggak` must be preserved because they can invert sentiment meaning. Neither preprocessing script removes stopwords aggressively.

## Emoji Handling Note
IndoBERT preprocessing avoids destructive emoji removal. SVM preprocessing converts emoji-like Unicode ranges into a simple `emoji` token so the signal is not silently discarded before feature extraction.

## Data Validation Plan
Validation checks include total rows, label distribution before and after relabeling, changed label count, audit candidate count, rating-3 changed count when `rating` exists, missing text counts, empty processed text counts, and text length statistics before and after cleaning.

## EDA Before vs After Plan
Preprocessing EDA compares label distributions before and after keyword relabeling and compares raw, IndoBERT, and SVM text length statistics. Figures are exported to `docs/figures/02_preprocessing/`.

## Processed Dataset Outputs
Generated local processed datasets:
- `datasets/processed/reviews_relabelled.csv`
- `datasets/processed/reviews_preprocessed_indobert.csv`
- `datasets/processed/reviews_preprocessed_svm.csv`
- `datasets/processed/reviews_final.csv`

These CSV files are generated artifacts and must remain ignored by Git.

## Frontend Metrics Outputs
Aggregate preprocessing metrics are exported to `datasets/outputs/eda/02_preprocessing/` for later dashboard visualization:
- `label_distribution_before_relabeling.csv`
- `label_distribution_before_relabeling.json`
- `label_distribution_after_relabeling.csv`
- `label_distribution_after_relabeling.json`
- `relabeling_summary.json`
- `text_length_before_after_cleaning.csv`
- `text_length_before_after_cleaning.json`
- `preprocessing_summary.json`

## Figure Outputs
Generated Phase 02 figures:
- `docs/figures/02_preprocessing/label_distribution_before_relabeling.png`
- `docs/figures/02_preprocessing/label_distribution_after_relabeling.png`
- `docs/figures/02_preprocessing/relabeling_change_summary.png`
- `docs/figures/02_preprocessing/text_length_before_after_cleaning.png`

## Reproducible Commands
Run from `ml-service/`:

```bash
uv run python scripts/relabel_by_keywords.py --input ../datasets/raw/reviews_raw_labeled.csv --output ../datasets/processed/reviews_relabelled.csv
uv run python scripts/preprocess_indobert.py --input ../datasets/processed/reviews_relabelled.csv --output ../datasets/processed/reviews_preprocessed_indobert.csv
uv run python scripts/preprocess_svm.py --input ../datasets/processed/reviews_preprocessed_indobert.csv --output ../datasets/processed/reviews_preprocessed_svm.csv
```

## Interpretation Notes
Interpret relabeling results as a conservative audit step, not as ground-truth manual annotation. Large relabeling shifts would indicate that rating-based sentiment labels need deeper review before model training.

## Limitations
Keyword rules are transparent and reproducible but cannot capture all Indonesian sentiment expressions, sarcasm, or context-dependent wording. This phase prepares data only; class balancing, dataset splits, tokenization, vectorizer fitting, and model training are deferred to later phases.

## Next Step
Continue to `03_indobert_sentiment_modeling.ipynb` for IndoBERT modeling preparation and training workflow design.